# Ablation-Only Study: Credit Card + Telco

Runs **only** the corrected group-ablation study for the two native-split datasets
(`credit_card`, `telco`).  Their MAIN model experiments are already valid and are
**never re-run** here — this notebook refreshes only `results/<ds>/<mode>/ablation/*`.

1. pulls the latest code from **https://github.com/nikhilwankhedee/churn** (`main`),
2. installs any missing dependencies from `requirements.txt`,
3. runs the pre-run validation gate (`src/preflight`) scoped to `credit_card,telco`
   (inputs present, SMOTE placement, every ablation group removes real columns),
4. recomputes each dataset's ablation on the **original** train matrix and the
   **SMOTE** train matrix (remove-real-columns, refit from scratch, no cached results).

**Attach these data inputs** (same two datasets the full sweep uses):

| Dataset | Mounted file the gate checks |
|---|---|
| `blastchar / telco-customer-churn` | `/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv` |
| `sakshigoyal7 / credit-card-customers` | `/kaggle/input/datasets/sakshigoyal7/credit-card-customers/BankChurners.csv` |

> Just hit **Run All**.  The gate stops Run All with a loud error if an input is missing.

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/nikhilwankhedee/churn.git"
REPO_BRANCH = "main"
UPDATE_REPO = True            # git pull the latest code on every run

# This notebook recomputes ONLY the ablation study for the two native-split
# datasets, in both train-matrix variants:
#   original (no SMOTE) -> results/<ds>/original/ablation/ablation_results.csv
#   smote               -> results/<ds>/smote/ablation/ablation_results_smote.csv
ABLATION_TARGETS = ["credit_card", "telco"]
WITH_SMOTE = True             # also run the ablation on the SMOTE train matrix

# ── Environment / paths ─────────────────────────────────────────────────
ON_KAGGLE = os.path.exists("/kaggle/working")
WORK_DIR = "/kaggle/working" if ON_KAGGLE else ("/content" if os.path.isdir("/content") else os.getcwd())
REPO_DIR = os.path.join(WORK_DIR, "churn")

print(f"Environment  : {'KAGGLE' if ON_KAGGLE else ('COLAB' if '/content' in WORK_DIR else 'LOCAL')}")
print(f"Code will go : {REPO_DIR}")


In [ ]:
# ── Clone / update the code from GitHub ───────────────────────────────────────
import io
import os
import sys
import shutil
import subprocess
import zipfile
import urllib.request


def git(cmd, cwd=None):
    print("$ git " + " ".join(cmd))
    subprocess.check_call(["git"] + cmd, cwd=cwd)


def sync_repo(repo_dir, url, branch, update=True):
    """Clone the repo, or pull the latest code if it already exists.
    Falls back to downloading GitHub's zipball when git is unavailable."""
    repo_dir = os.path.abspath(repo_dir)
    parent = os.path.dirname(repo_dir)

    if os.path.isdir(os.path.join(repo_dir, ".git")):
        if not update:
            print(f"→ Using existing repo at {repo_dir} (UPDATE_REPO=False)")
        else:
            try:
                git(["fetch", "origin"], repo_dir)
                git(["checkout", "--quiet", branch], repo_dir)
                git(["pull", "--rebase", "origin", branch], repo_dir)
                print("→ Repo updated to latest.")
            except Exception as exc:
                # network hiccup — keep the local checkout, it still works offline
                print(f"→ Update failed ({exc}); reusing existing checkout.")
        return repo_dir

    try:
        git(["clone", "--branch", branch, url, repo_dir])
        print(f"→ Cloned {url} → {repo_dir}")
    except Exception:
        # git missing or blocked (e.g. no git binary) — download the zipball instead
        print("git clone failed — downloading GitHub zipball instead…")
        if os.path.isdir(repo_dir):
            shutil.rmtree(repo_dir)  # leftover from a partial clone
        os.makedirs(parent, exist_ok=True)
        zip_url = url.replace(".git", "") + f"/archive/refs/heads/{branch}.zip"
        with urllib.request.urlopen(zip_url, timeout=120) as resp:
            data = resp.read()
        extract_dir = os.path.join(parent, "__churn_extract__")
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            zf.extractall(extract_dir)
        extracted = [os.path.join(extract_dir, d) for d in os.listdir(extract_dir)
                     if os.path.isdir(os.path.join(extract_dir, d))]
        shutil.move(extracted[0], repo_dir)
        shutil.rmtree(extract_dir)
        print(f"→ Downloaded {zip_url} → {repo_dir}")
    return repo_dir


REPO_DIR = sync_repo(REPO_DIR, REPO_URL, REPO_BRANCH, update=UPDATE_REPO)

In [ ]:
# ── Install any missing dependencies (from the repo's requirements.txt) ──────
from importlib.util import find_spec

MODULES = {
    "pandas": "pandas", "numpy": "numpy", "scikit-learn": "sklearn",
    "xgboost": "xgboost", "lightgbm": "lightgbm", "imbalanced-learn": "imblearn",
    "shap": "shap", "matplotlib": "matplotlib", "seaborn": "seaborn",
    "scipy": "scipy", "pingouin": "pingouin", "statsmodels": "statsmodels",
    "joblib": "joblib", "tqdm": "tqdm", "pyyaml": "yaml", "typer": "typer",
    "rich": "rich", "openpyxl": "openpyxl",
}
SKIP = {"jupyter", "ipykernel"}  # already provided by the notebook host

missing = []
with open(os.path.join(REPO_DIR, "requirements.txt")) as fh:
    for line in fh:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        name = line.split(">=")[0].split("==")[0].split("[")[0].strip()
        if name.lower() in SKIP:
            continue
        mod = MODULES.get(name.lower(), name.lower().replace("-", "_"))
        if find_spec(mod) is None:
            missing.append(name)

if missing:
    print(f"Installing {len(missing)} missing package(s): {missing}")
    cmd = [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        # PEP 668 'externally managed environment' (e.g. Debian/Ubuntu system
        # python) — retry with the OS-sanctioned override flag
        print("pip refused — retrying with --break-system-packages")
        subprocess.check_call(cmd + ["--break-system-packages"])
else:
    print("All pipeline dependencies are already installed.")

In [ ]:
# ── Add the cloned code to the path and verify the framework imports ──
os.chdir(REPO_DIR)
os.environ["PYTHONPATH"] = REPO_DIR + os.pathsep + os.environ.get("PYTHONPATH", "")

from src.config import ON_KAGGLE, PROJECT_ROOT, CREDIT_CARD_FILE, TELCO_FILE
from src.datasets import list_datasets

print(f"On Kaggle    : {ON_KAGGLE}")
print(f"Project root : {PROJECT_ROOT}")
print(f"Reg datasets : {', '.join(list_datasets())}")

# The two data inputs this notebook needs (ablation train matrices):
for label, path in [("credit_card", CREDIT_CARD_FILE), ("telco", TELCO_FILE)]:
    present = os.path.isfile(path)
    print(f"  input {label:12s}: {'present' if present else 'MISSING'}  {path}")
    if not present:
        print(f"    --> attach the dataset input '...' that provides {path}")

try:
    rev = subprocess.check_output(["git", "-C", REPO_DIR, "log", "-1",
                                   "--format=%h  %cs  %s"], text=True).strip()
    print(f"Code version : {rev}")
except Exception:
    pass


In [ ]:
# ── Pre-run validation gate (scoped to credit_card + telco) ───────────
import pandas as pd
from IPython.display import display

from src.preflight import run_preflight, gate_all_passed

report = run_preflight(ABLATION_TARGETS)
display(report)
if not gate_all_passed(report):
    raise RuntimeError(
        "PRE-RUN VALIDATION FAILED — fix the FAIL rows above before running the ablation."
    )
print("GATE RESULT: PASS — inputs, SMOTE placement and ablation-group resolution are valid.")


In [ ]:
# ── Run the corrected ablation for credit_card + telco ───────────────
import time
from src.run_ablation_only import run_ablation_only

modes = [(False, "original")] + ([(True, "smote")] if WITH_SMOTE else [])
for ds in ABLATION_TARGETS:
    for use_smote, mode in modes:
        t0 = time.time()
        try:
            df = run_ablation_only(ds, use_smote=use_smote)
            n_rows = len(df)
            coverage = df["feature_set"].nunique() if "feature_set" in df else 0
            print(f"  ablation {ds:12s} {mode:8s} OK  {n_rows} rows "
                  f"({coverage} feature sets / model)  {time.time()-t0:7.1f}s")
            display(df[["model", "feature_set", "n_removed", "mean_roc_auc", "std_roc_auc"]]
                      .sort_values(["model", "feature_set"]))
        except Exception as exc:
            print(f"  ablation {ds:12s} {mode:8s} FAILED: {str(exc)[:200]}")
print("\nDone. Ablation CSVs live in results/{credit_card,telco}/<mode>/ablation/.")


In [ ]:
# ── Inspect the produced ablation artefacts ───────────────────────────
import os
from src.config import RESULTS_DIR

for ds in ABLATION_TARGETS:
    for mode, suffix in [("original", ""), ("smote", "_smote")]:
        p = os.path.join(RESULTS_DIR, ds, mode, "ablation", f"ablation_results{suffix}.csv")
        if os.path.isfile(p):
            d = pd.read_csv(p)
            print(f"{ds}/{mode:8s} ablation_results{suffix}.csv  ({len(d)} rows)")
            display(d.sort_values(["model", "feature_set"]))
        else:
            print(f"{ds}/{mode:8s} missing: {p}")


## What this notebook did / what to check

- **Only** `results/<dataset>/<mode>/ablation/ablation_results.csv` / `ablation_results_smote.csv` were refreshed
  (`dataset ∈ {credit_card, telco}`, `mode ∈ {original, smote}`) plus the matching
  plot under `figures/<dataset>/<mode>/model_evaluation/ablation_results_smote.png`.
- The **MAIN** model experiments, metrics, risk models, SHAP and segmentation
  artefacts for credit_card/telco are fully untouched.
- Verify the fix: `mean_roc_auc` is **no longer byte-identical across `without_*` rows**
  and every row reports `n_removed > 0` (real columns dropped, refit from scratch).
- The pre-run gate report is saved at `results/preflight/preflight_report_<timestamp>.csv`.


## Notes

- **Inputs to attach:** `blastchar/telco-customer-churn` and
  `sakshigoyal7/credit-card-customers`.  The gate FAILs loudly (and stops Run All)
  if either expected file is missing.
- **No more zip uploads:** code is pulled live from
  https://github.com/nikhilwankhedee/churn on every run (`UPDATE_REPO = True`).
- **Why ablation-only?** Previous credit_card/telco ablations removed zero columns
  (generic group names did not match the native predictors), producing identical
  AUC for every group.  The dataset-aware group maps + hard assertions make it fail
  loud instead of silent-invalid.  See the commit + `churn_pipeline_audit_report_2026-08-30.md`.
- **SMOTE placement:** applied only to the training fold, post split + feature
  engineering; the test fold is never resampled (verified by the gate).
- **To extend:** edit `ABLATION_TARGETS` / `WITH_SMOTE` in the Settings cell, or run
  `python -m src.run_ablation_only credit_card telco [--with-smote]` from a Kaggle shell.
